# PyCaffe (caffe-slim) 快速上手

本 Notebook 演示如何在 `caffe-cpu:pycaffe-jupyter-ssh` 容器中使用 PyCaffe 进行：
1. 环境验证与基本信息查看
2. 图像预处理（手动模拟 Caffe Transformer 流程）
3. 构建一个简单的卷积网络并执行前向推理
4. 可视化中间层特征图

> **注意**：本镜像使用的是 caffe-slim 版本，API 与原版 BVLC/Caffe 略有不同：
> - `net.set_input_data(name, array)` 设置输入数据
> - `net.blob_data(name)` 获取中间/输出 blob
> - `net.forward()` 执行前向传播（返回 None，结果通过 blob_data 获取）
> - 没有 `caffe.io` 模块，图像预处理需手动完成

In [ ]:
import os
os.environ['GLOG_minloglevel'] = '2'  # 抑制 C++ glog 输出

import sys
import numpy as np
import caffe

print('=' * 50)
print('PyCaffe 环境信息')
print('=' * 50)
print(f'Python 版本: {sys.version}')
print(f'NumPy 版本:  {np.__version__}')
print(f'Caffe 版本:  {caffe.version()}')
print(f'Caffe 路径:  {caffe.__file__}')
print(f'可用层类型数: {len(caffe.layer_type_list())}')
print(f'常用层类型: {caffe.layer_type_list()[:15]}')
print()

caffe.set_mode_cpu()
print('[OK] CPU 模式已设置')

## 1. 图像预处理流程

Caffe 标准的图像预处理步骤（手动实现 `caffe.io.Transformer` 的核心功能）：
1. 加载图像 → numpy array (H, W, C)，RGB 通道顺序
2. Resize 到网络输入尺寸
3. 转换为 float32，像素值缩放到 [0, 1]
4. 减去 ImageNet 均值（可选，此处用 0 均值演示）
5. 通道顺序从 HWC 转为 CHW
6. 添加 batch 维度 → (1, C, H, W)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def generate_test_image(height=32, width=32, seed=42):
    """生成一个合成测试图像（彩色渐变+噪声），模拟 CIFAR-10 尺寸"""
    rng = np.random.RandomState(seed)
    # 创建彩色渐变图案
    x = np.linspace(0, 1, width)
    y = np.linspace(0, 1, height)
    xx, yy = np.meshgrid(x, y)
    r = (xx * 255).astype(np.uint8)
    g = (yy * 255).astype(np.uint8)
    b = ((1 - xx) * (1 - yy) * 255).astype(np.uint8)
    img = np.stack([r, g, b], axis=-1)
    # 添加少量噪声
    noise = rng.randint(-20, 20, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(img, mode='RGB')

def preprocess_caffe(pil_image, input_height=32, input_width=32, mean=None):
    """Caffe 标准预处理流程
    
    Args:
        pil_image: PIL Image 对象 (RGB)
        input_height: 网络输入高度
        input_width: 网络输入宽度
        mean: 均值数组，shape=(3,)，默认不减均值
    
    Returns:
        numpy array, shape=(1, 3, H, W), dtype=float32
    """
    # Step 1: Resize
    img = pil_image.resize((input_width, input_height), Image.BILINEAR)
    
    # Step 2: 转为 float32, 缩放到 [0, 1]
    arr = np.array(img, dtype=np.float32) / 255.0
    
    # Step 3: 减均值
    if mean is not None:
        arr = arr - np.array(mean, dtype=np.float32).reshape(1, 1, 3)
    
    # Step 4: HWC → CHW
    arr = arr.transpose(2, 0, 1)
    
    # Step 5: 添加 batch 维度
    arr = arr[np.newaxis, ...]
    
    return arr.astype(np.float32)

# 生成测试图像
test_img = generate_test_image(64, 64)
print(f'原始图像尺寸: {test_img.size}')

# 预处理
input_blob = preprocess_caffe(test_img, input_height=32, input_width=32)
print(f'预处理后 blob shape: {input_blob.shape}')
print(f'预处理后 blob dtype: {input_blob.dtype}')
print(f'值范围: [{input_blob.min():.4f}, {input_blob.max():.4f}]')
print(f'均值: {input_blob.mean():.4f}')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(test_img)
axes[0].set_title('Original (64x64)')
axes[0].axis('off')

# 显示预处理后的第一个通道
axes[1].imshow(input_blob[0, 0], cmap='gray')
axes[1].set_title('Preprocessed R channel (32x32)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. 构建网络并执行前向推理

创建一个简单的 CNN（Conv→ReLU→Pool→FC→Softmax），演示从 prototxt 字符串定义网络、加载数据、执行推理的完整流程。

In [ ]:
import tempfile

# 定义网络结构（prototxt 格式）
PROTOTXT = """
name: "demo_cnn"
layer {
  name: "data"
  type: "Input"
  top: "data"
  input_param { shape { dim: 1 dim: 3 dim: 32 dim: 32 } }
}
layer {
  name: "conv1"
  type: "Convolution"
  bottom: "data"
  top: "conv1"
  convolution_param {
    num_output: 16
    kernel_size: 3
    stride: 1
    pad: 1
    weight_filler { type: "xavier" }
    bias_filler { type: "constant" }
  }
}
layer {
  name: "relu1"
  type: "ReLU"
  bottom: "conv1"
  top: "conv1"
}
layer {
  name: "pool1"
  type: "Pooling"
  bottom: "conv1"
  top: "pool1"
  pooling_param { pool: MAX kernel_size: 2 stride: 2 }
}
layer {
  name: "conv2"
  type: "Convolution"
  bottom: "pool1"
  top: "conv2"
  convolution_param {
    num_output: 32
    kernel_size: 3
    stride: 1
    pad: 1
    weight_filler { type: "xavier" }
    bias_filler { type: "constant" }
  }
}
layer {
  name: "relu2"
  type: "ReLU"
  bottom: "conv2"
  top: "conv2"
}
layer {
  name: "pool2"
  type: "Pooling"
  bottom: "conv2"
  top: "pool2"
  pooling_param { pool: MAX kernel_size: 2 stride: 2 }
}
layer {
  name: "fc1"
  type: "InnerProduct"
  bottom: "pool2"
  top: "fc1"
  inner_product_param {
    num_output: 128
    weight_filler { type: "xavier" }
    bias_filler { type: "constant" }
  }
}
layer {
  name: "relu3"
  type: "ReLU"
  bottom: "fc1"
  top: "fc1"
}
layer {
  name: "fc2"
  type: "InnerProduct"
  bottom: "fc1"
  top: "fc2"
  inner_product_param {
    num_output: 10
    weight_filler { type: "xavier" }
    bias_filler { type: "constant" }
  }
}
layer {
  name: "prob"
  type: "Softmax"
  bottom: "fc2"
  top: "prob"
}
"""

# 写入临时 prototxt 文件并创建网络
with tempfile.NamedTemporaryFile(mode='w', suffix='.prototxt', delete=False) as f:
    f.write(PROTOTXT)
    prototxt_path = f.name

caffe.set_random_seed(42)
net = caffe.Net(prototxt_path, caffe.TEST)

print('网络结构信息:')
print(f'  输入 blobs: {net.inputs}')
print(f'  输出 blobs: {net.outputs}')
print(f'  所有 blobs:')
for name in net.blob_names:
    print(f'    {name:12s} -> shape={net.blob_shape(name)}')

# 设置输入并执行推理
net.set_input_data('data', input_blob)
net.forward()

# 获取结果
prob = net.blob_data('prob')[0]  # 去掉 batch 维度
pred_class = prob.argmax()

print(f'\n推理结果:')
print(f'  预测类别: {pred_class}')
print(f'  置信度:   {prob[pred_class]:.4f}')
print(f'  Softmax 校验 (sum≈1.0): {prob.sum():.6f}')

## 3. 可视化各层特征图

查看卷积层和池化层的输出特征图，直观理解网络各层学到的模式。

In [ ]:
def visualize_feature_maps(net, blob_name, num_channels=16, figsize=(14, 4)):
    """可视化指定 blob 的前 num_channels 个通道"""
    feat = net.blob_data(blob_name)[0]  # (C, H, W)
    n = min(num_channels, feat.shape[0])
    cols = min(8, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i in range(n):
        im = axes[i].imshow(feat[i], cmap='viridis')
        axes[i].set_title(f'ch{i}', fontsize=8)
        axes[i].axis('off')
    for i in range(n, len(axes)):
        axes[i].axis('off')
    plt.suptitle(f'Feature maps: {blob_name}  shape={feat.shape}', fontsize=12)
    plt.tight_layout()
    plt.show()

# 可视化 conv1 (第一个卷积层) — 低级特征
visualize_feature_maps(net, 'conv1', num_channels=16)

# 可视化 pool1
visualize_feature_maps(net, 'pool1', num_channels=16)

# 可视化 conv2 (第二个卷积层) — 更高级特征
visualize_feature_maps(net, 'conv2', num_channels=32, figsize=(14, 8))

## 4. 批量推理测试

对多张图像进行批量预处理和推理。

In [ ]:
# 生成多张测试图像
num_samples = 5
batch_data = []
fig, axes = plt.subplots(1, num_samples, figsize=(12, 3))

for i in range(num_samples):
    img = generate_test_image(64, 64, seed=i * 100)
    blob = preprocess_caffe(img, input_height=32, input_width=32)
    batch_data.append(blob[0])  # 去掉临时 batch 维度
    axes[i].imshow(img)
    axes[i].set_title(f'Sample {i}')
    axes[i].axis('off')

plt.suptitle('Test Images', fontsize=12)
plt.tight_layout()
plt.show()

# 注意：当前网络输入 shape 固定为 batch=1
# 若要支持 batch>1，需修改 prototxt 中 input_shape 的 dim=1 为更大值
# 这里逐一推理演示
print('单张逐一推理结果:')
for i in range(num_samples):
    single_blob = batch_data[i][np.newaxis, ...]  # 添加 batch 维度
    net.set_input_data('data', single_blob.astype(np.float32))
    net.forward()
    prob = net.blob_data('prob')[0]
    top3 = np.argsort(prob)[-3:][::-1]
    print(f'  Sample {i}: pred={top3[0]}, conf={prob[top3[0]]:.4f}, top3={list(top3)}')

print('\n[OK] PyCaffe 演示完成！')